# Baseline RAG — SEC Filings

**Pipeline**: Query → Dense Retrieval (top-5) → LLM Generation

The simplest possible RAG baseline:
- Embed the raw query with `all-mpnet-base-v2`
- Retrieve top-5 chunks from the pre-built ChromaDB vector store
- Pass context + question to `llama-3.1-8b-instant` (dev) or `llama-4-scout-17b` (final)
- No query transformation, no metadata filtering, no reranking

Results saved to `./results/baseline_results.csv` for comparison with Advanced RAG.

In [1]:
print('hi')

hi


In [3]:
# Uncomment to install dependencies
# !pip install sentence-transformers chromadb groq python-dotenv pandas tqdm

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached chromadb-1.5.5-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached groq-1.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached torch-2.10.0-cp311-cp311-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached regex-2026.2.28-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dm-tree 0.1.9 requires absl-py>=0.6.1, which is not installed.
dm-tree 0.1.9 requires wrapt>=1.11.2, which is not installed.
tensorflow-datasets 4.9.9 requires absl-py, which is not installed.
tensorflow-datasets 4.9.9 requires pyarrow, which is not installed.
tensorflow-datasets 4.9.9 requires termcolor, which is not installed.
tensorflow-datasets 4.9.9 requires toml, which is not installed.
tensorflow-datasets 4.9.9 requires wrapt, which is not installed.
tensorflow-metadata 1.17.2 requires absl-py<3.0.0,>=0.9, which is not installed.


## 1. Configuration

In [2]:
CONFIG = {
    # --- Paths (relative to this notebook) ---
    "chroma_db_path": "../sec_rag_team_share/chroma_db",
    "chunks_path":    "../sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl",
    "eval_path":      "../sec_rag_team_share/evaluation/GenAI Eval QA.csv",
    "results_dir":    "./results",

    # --- Models ---
    "dense_model_name":   "sentence-transformers/all-mpnet-base-v2",
    # Execution profile: 'dev' or 'final' (same model — kept for parity with Advanced RAG)
    "execution_profile":  "dev",
    "gemini_dev_model":   "gemini-2.5-flash-lite",
    "gemini_final_model": "gemini-2.5-flash-lite",
    "judge_model":        "gemini-2.5-flash-lite",

    # --- Retrieval ---
    "dense_top_k":       5,
    "max_context_chars": 7000,

    # --- Generation ---
    "temperature_generator": 0.2,
    "temperature_judge":     0.1,   # 0.0 causes empty responses in Gemini via LiteLLM
    "max_tokens_answer":     512,
    "max_tokens_judge":      256,

    # --- Cost tracking (gemini-2.5-flash-lite) ---
    "gemini_cost_input_per_1m":  0.10,
    "gemini_cost_output_per_1m": 0.40,

    # --- Evaluation ---
    "eval_split": "dev",

    # --- Rate limiting (Gemini free tier: 10 RPM) ---
    # Baseline makes 3 LLM calls per question: generate + judge_corr + judge_faith
    "max_rpm":                    10,
    "inter_question_sleep_sec":   20,
    "llm_max_retries":            3,
    "llm_retry_base_delay_sec":   5,
}

## 2. Imports & Setup

In [3]:
import os
import json
import time
import threading
from collections import deque, Counter as _Counter
from pathlib import Path
from typing import Optional
import re as _re

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel, Field

import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types as genai_types

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in your .env file"

LLM_MODEL = (
    CONFIG["gemini_final_model"]
    if CONFIG["execution_profile"] == "final"
    else CONFIG["gemini_dev_model"]
)
print(f"Execution profile : {CONFIG['execution_profile']}")
print(f"LLM model         : {LLM_MODEL}")

c:\Users\jolen\anaconda3\envs\genai_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Execution profile : dev
LLM model         : gemini-2.5-flash-lite


## 3. Load Models

In [4]:
print("Loading embedding model...")
embed_model = SentenceTransformer(CONFIG["dense_model_name"])
print(f"  ok {CONFIG['dense_model_name']}")

genai_client = genai.Client(api_key=GEMINI_API_KEY)
print(f"  ok Gemini client ready (model: {LLM_MODEL})")


# ── Rate Limiter ───────────────────────────────────────────────────────────────
class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self._ts: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self._ts and now - self._ts[0] >= 60.0:
                self._ts.popleft()
            if len(self._ts) >= self.max_rpm:
                sleep_for = 60.0 - (now - self._ts[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] sleeping {sleep_for:.1f}s ...")
                    time.sleep(sleep_for)
            self._ts.append(time.time())

rate_limiter = RateLimiter(CONFIG["max_rpm"])


# ── Per-Question Token Accumulator ─────────────────────────────────────────────
_question_tokens: dict = {}

def _reset_question_tokens() -> None:
    global _question_tokens
    _question_tokens = {}

def _add_tokens(step: str, input_tok: int, output_tok: int) -> None:
    if step not in _question_tokens:
        _question_tokens[step] = {'input': 0, 'output': 0}
    _question_tokens[step]['input']  += int(input_tok  or 0)
    _question_tokens[step]['output'] += int(output_tok or 0)

def _get_question_token_totals() -> tuple:
    total_in  = sum(v['input']  for v in _question_tokens.values())
    total_out = sum(v['output'] for v in _question_tokens.values())
    return total_in, total_out

def _estimate_cost_usd(total_input: int, total_output: int) -> float:
    rate_in  = CONFIG.get('gemini_cost_input_per_1m',  0.10) / 1_000_000
    rate_out = CONFIG.get('gemini_cost_output_per_1m', 0.40) / 1_000_000
    return total_input * rate_in + total_output * rate_out


# ── LLM Call ───────────────────────────────────────────────────────────────────
def call_llm(
    messages: list,
    model: str = None,
    temperature: float = 0.2,
    max_tokens: int = 512,
    json_mode: bool = False,
    step: str = 'generate',
) -> str:
    """Call Gemini with retry. Accumulates tokens to _question_tokens[step]."""
    model = model or LLM_MODEL
    system_instruction = None
    user_parts = []
    for msg in messages:
        if msg["role"] == "system":
            system_instruction = msg["content"]
        else:
            user_parts.append(msg["content"])
    contents = "\n\n".join(user_parts) if user_parts else ""

    cfg_kwargs = dict(temperature=temperature, max_output_tokens=max_tokens)
    if json_mode:
        cfg_kwargs["response_mime_type"] = "application/json"
    if system_instruction:
        cfg_kwargs["system_instruction"] = system_instruction

    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            resp = genai_client.models.generate_content(
                model=model,
                contents=contents,
                config=genai_types.GenerateContentConfig(**cfg_kwargs),
            )
            if step and resp.usage_metadata:
                _add_tokens(step,
                    resp.usage_metadata.prompt_token_count     or 0,
                    resp.usage_metadata.candidates_token_count or 0)
            return resp.text.strip()
        except Exception as e:
            delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
            print(f"  [WARN] attempt {attempt+1}/{CONFIG['llm_max_retries']} failed ({e}). Retrying in {delay}s...")
            time.sleep(delay)
    raise RuntimeError(f"LLM call failed after {CONFIG['llm_max_retries']} attempts")


# ── Heuristic Scoring Utilities ────────────────────────────────────────────────
def _tokenize(text: str) -> list:
    return _re.sub(r'[^\w\s]', '', text.lower()).split()

def token_f1_score(pred: str, ref: str) -> float:
    if not pred or not ref:
        return 0.0
    pred_toks = _tokenize(pred)
    ref_toks  = _tokenize(ref)
    if not pred_toks or not ref_toks:
        return 0.0
    common = sum((_Counter(pred_toks) & _Counter(ref_toks)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_toks)
    recall    = common / len(ref_toks)
    return 2 * precision * recall / (precision + recall)

def numerical_accuracy_score(pred: str, ref: str) -> Optional[float]:
    nums_ref  = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', ref)]
    nums_pred = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', pred)]
    if not nums_ref:
        return None
    hits = sum(
        1 for r in nums_ref
        if any(abs(p - r) / (abs(r) + 1e-9) < 0.01 for p in nums_pred)
    )
    return hits / len(nums_ref)

def compute_decision_from_text(answer_text: str) -> str:
    lower = answer_text.lower()
    refusal_phrases = [
        'insufficient', 'not contain', 'not available', 'cannot find',
        "don't have", 'no information', 'not provided', 'unable to',
        'not enough', 'not present', 'not found', 'insufficient data',
    ]
    return 'refuse' if any(p in lower for p in refusal_phrases) else 'answer'


# ── Structured Judge (matches CrewAI) ─────────────────────────────────────────
class JudgeOutput(BaseModel):
    score:          float = Field(default=0.0, description='Score 0.0-1.0')
    claims_covered: float = Field(default=0.0, description='Fraction of key facts covered 0.0-1.0')
    reason:         str   = Field(default='',  description='Short explanation')

def _build_correctness_prompt(question: str, reference_answer: str, candidate_answer: str) -> str:
    return (
        'Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. '
        '1 = correct value, correct units, correct period. '
        '0 = wrong number, wrong company, wrong period, or completely missing. '
        'Give partial credit for answers close but with unit errors. '
        'Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Reference answer:\n{reference_answer}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )

def _build_faithfulness_prompt(question: str, context: str, candidate_answer: str) -> str:
    return (
        'Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. '
        '1 = every number and claim is directly supported by the context. '
        '0 = answer contains numbers or claims not present in the context (hallucinated). '
        'Also set claims_covered: fraction of claims in the candidate supported by the context. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Retrieved context:\n{context}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )

def _safe_judge_call(prompt_text: str, task_name: str) -> JudgeOutput:
    _fb = JudgeOutput(score=0.0, claims_covered=0.0, reason='judge fallback — all attempts failed')
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            response = genai_client.models.generate_content(
                model=CONFIG["judge_model"],
                contents=prompt_text,
                config=genai_types.GenerateContentConfig(
                    response_mime_type='application/json',
                    response_schema=JudgeOutput,
                    temperature=CONFIG.get('temperature_judge', 0.1),
                ),
            )
            if response.usage_metadata:
                _add_tokens(task_name,
                    response.usage_metadata.prompt_token_count     or 0,
                    response.usage_metadata.candidates_token_count or 0)
            result = response.parsed
            if result is None:
                result = JudgeOutput(**json.loads(response.text))
            return result
        except Exception as e:
            print(f"  [WARN] Judge ({task_name}) attempt {attempt+1} failed: {e}")
            if attempt < CONFIG["llm_max_retries"] - 1:
                time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return _fb

def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> tuple:
    result = _safe_judge_call(
        _build_correctness_prompt(question, reference_answer, candidate_answer),
        task_name='judge_correctness',
    )
    return (max(0.0, min(1.0, float(result.score))),
            max(0.0, min(1.0, float(result.claims_covered))),
            result.reason)

def llm_judge_faithfulness(question: str, context: str, candidate_answer: str) -> tuple:
    result = _safe_judge_call(
        _build_faithfulness_prompt(question, context[:CONFIG.get('max_context_chars', 7000)], candidate_answer),
        task_name='judge_faithfulness',
    )
    return (max(0.0, min(1.0, float(result.score))),
            max(0.0, min(1.0, float(result.claims_covered))),
            result.reason)

print("Client, rate limiter, token tracking, heuristics, and judge ready.")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7700.02it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ok sentence-transformers/all-mpnet-base-v2
  ok Gemini client ready (model: gemini-2.5-flash-lite)
Client, rate limiter, token tracking, heuristics, and judge ready.


## 4. Load ChromaDB Vector Store

In [5]:
print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_db_path"])
collections = chroma_client.list_collections()
print(f"  Available collections: {[c.name for c in collections]}")
collection = chroma_client.get_collection(collections[0].name)
print(f"  ok Using '{collections[0].name}'  ({collection.count():,} chunks)")

Connecting to ChromaDB...
  Available collections: ['sec_filings']
  ok Using 'sec_filings'  (12,725 chunks)


## 5. Retrieval — Dense Search Only

In [ ]:
# def retrieve(query: str, top_k: int = None) -> list:
#     """
#     Dense retrieval: embed query -> cosine nearest-neighbor in ChromaDB.
#     Returns list of dicts: text, metadata, score, chunk_id.
#     """
#     k = top_k or CONFIG["dense_top_k"]
#     query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
#     results = collection.query(
#         query_embeddings=[query_vec],
#         n_results=min(k, collection.count()),
#         include=["documents", "metadatas", "distances", "ids"],
#     )
#     chunks = []
#     for doc, meta, dist, cid in zip(
#         results["documents"][0],
#         results["metadatas"][0],
#         results["distances"][0],
#         results["ids"][0],
#     ):
#         chunks.append({
#             "text":     doc,
#             "metadata": meta,
#             "score":    float(1.0 - dist),  # cosine distance -> similarity
#             "chunk_id": cid,
#         })
#     return chunks

In [6]:
def retrieve(query: str, top_k: int = None) -> list:
    """
    Dense retrieval: embed query -> cosine nearest-neighbor in ChromaDB.
    Returns list of dicts: text, metadata, score, chunk_id.
    """
    k = top_k or CONFIG["dense_top_k"]
    query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=min(k, collection.count()),
        include=["documents", "metadatas", "distances"],  
    )

    chunks = []
    for doc, meta, dist, cid in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
        results["ids"][0], 
    ):
        chunks.append({
            "text": doc,
            "metadata": meta,
            "score": float(1.0 - dist),  # cosine distance -> similarity
            "chunk_id": cid,
        })

    return chunks

## 6. Generation

In [7]:
GENERATOR_SYSTEM = (
    "You are a financial analyst answering questions using only the retrieved filing context. "
    "Be precise with numbers -- include units (millions, billions, %), fiscal periods, and line item names. "
    "If the context does not contain the answer, say so explicitly rather than estimating."
)


def format_context(chunks: list, max_chars: int = None) -> str:
    """Format retrieved chunks into a numbered context string."""
    max_chars = max_chars or CONFIG["max_context_chars"]
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        header = f"[{i}] {m.get('ticker','?')} {m.get('form_type','?')} {m.get('filing_year','?')}"
        parts.append(f"{header}\n{c['text']}")
    context = "\n\n---\n\n".join(parts)
    return context[:max_chars]


def generate(query: str, chunks: list) -> str:
    """Generate an answer from retrieved chunks."""
    context  = format_context(chunks)
    user_msg = f"Question:\n{query}\n\nRetrieved context:\n{context}"
    return call_llm(
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        temperature=CONFIG["temperature_generator"],
        max_tokens=CONFIG["max_tokens_answer"],
    )

## 7. Full Baseline RAG Pipeline

In [8]:
def baseline_rag(query: str) -> dict:
    """
    Baseline RAG: Dense Retrieval -> LLM Generation.
    Returns: query, answer, chunks, context.
    """
    chunks  = retrieve(query)
    context = format_context(chunks)
    answer  = generate(query, chunks)

    # print(f'chunks: {chunks}')
    # print('\n\n\---------------')

    # print(f'context: {context}')
    # print('\n\n\---------------')

    # print(f'answer: {answer}')
    return {"query": query, "answer": answer, "chunks": chunks, "context": context}

### Quick Demo

In [9]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
# demo_q = "What was CVX's total net revenue for fiscal year 2023?"

result = baseline_rag(demo_q)

print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Retrieved {len(result['chunks'])} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  score={c['score']:.3f}")

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: This document does not contain the answer to this question.

Retrieved 5 chunks:
  [1] JPM     10-Q   year=2023  score=0.075
  [2] JPM     10-Q   year=2024  score=0.074
  [3] JPM     10-Q   year=2024  score=0.070
  [4] JPM     10-K   year=2025  score=0.069
  [5] JPM     10-Q   year=2023  score=0.069


## 8. Evaluation (LLM-as-Judge)

Uses the same judge prompts as `langgraph_agentic_rag_sec_v2`.
- **Correctness** (0-1): does the answer match the reference on values, units, fiscal period?
- **Faithfulness** (0-1): is every claim grounded in the retrieved context?
- **Claims covered** (0-1): fraction of reference facts present in the candidate.
- **Refusal accuracy**: for adversarial questions, did the model correctly say it cannot answer?

In [10]:
eval_df = pd.read_csv(CONFIG["eval_path"])
eval_df = eval_df[eval_df["split"] == CONFIG["eval_split"]].reset_index(drop=True)

print(f"Evaluation split : '{CONFIG['eval_split']}'  ({len(eval_df)} questions)")
print(eval_df["question_type"].value_counts().to_string())

Evaluation split : 'dev'  (20 questions)
question_type
extractive     5
paraphrased    5
reasoning      5
adversarial    5


In [13]:
results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating baseline"):
    question          = str(row["question"])
    reference_answer  = str(row["expected_answer"]).strip()
    question_id       = row.get("question_id", idx)
    question_type     = row.get("question_type", "unknown")
    company           = row.get("company", "")
    ticker            = row.get("ticker", "")
    form_type         = row.get("form_type", "")
    filing_year       = row.get("filing_year", None)
    expected_decision = str(row.get("expected_decision", "answer")).lower().strip()

    print(f'\nQ{idx+1}/{len(eval_df)} [{question_type}]  {ticker}  {filing_year}')

    # ── Reset token accumulator ────────────────────────────────────────────────
    _reset_question_tokens()

    # ── Run pipeline ──────────────────────────────────────────────────────────
    t0 = time.time()
    rag = baseline_rag(question)
    latency_sec  = round(time.time() - t0, 2)
    final_answer = rag["answer"]
    context      = rag["context"]

    # ── Heuristic scores ───────────────────────────────────────────────────────
    t_f1    = token_f1_score(final_answer, reference_answer) if reference_answer else None
    num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None
    predicted_decision = compute_decision_from_text(final_answer)
    decision_accuracy  = 1.0 if predicted_decision == expected_decision else 0.0

    # ── LLM Judge ──────────────────────────────────────────────────────────────
    c_score, c_covered, c_reason = llm_judge_correctness(question, reference_answer, final_answer)
    f_score, _,         f_reason = llm_judge_faithfulness(question, context, final_answer)

    # ── Token / cost summary ───────────────────────────────────────────────────
    gen_tok   = _question_tokens.get('generate',          {'input': 0, 'output': 0})
    corr_tok  = _question_tokens.get('judge_correctness', {'input': 0, 'output': 0})
    faith_tok = _question_tokens.get('judge_faithfulness',{'input': 0, 'output': 0})
    total_in, total_out = _get_question_token_totals()
    est_cost = _estimate_cost_usd(total_in, total_out)

    f1_str = f"{t_f1:.2f}" if t_f1 is not None else "N/A"
    print(f'  corr={c_score:.2f}  faith={f_score:.2f}  f1={f1_str}  '
          f'tokens={total_in}in/{total_out}out  est=${est_cost:.5f}')

    results.append({
        # Question metadata
        "question_id":              question_id,
        "question":                 question,
        "question_type":            question_type,
        "company":                  company,
        "ticker":                   ticker,
        "form_type":                form_type,
        "filing_year":              filing_year,
        "reference_answer":         reference_answer,
        "expected_decision":        expected_decision,
        # Answer
        "final_answer":             final_answer,
        "pipeline":                 "baseline_rag",
        # Pipeline metadata
        "n_chunks":                 len(rag["chunks"]),
        "latency_sec":              latency_sec,
        # Heuristic scores
        "token_f1":                 t_f1,
        "numerical_accuracy":       num_acc,
        "decision_accuracy":        decision_accuracy,
        "predicted_decision":       predicted_decision,
        # LLM Judge
        "llm_correctness_score":    c_score,
        "llm_claims_covered":       c_covered,
        "llm_correctness_reason":   c_reason,
        "llm_faithfulness_score":   f_score,
        "llm_faithfulness_reason":  f_reason,
        # Model logging
        "model_generator":          LLM_MODEL,
        "model_judge_correctness":  CONFIG["judge_model"],
        "model_judge_faithfulness": CONFIG["judge_model"],
        # Token & cost
        "tokens_generate_input":    gen_tok["input"],
        "tokens_generate_output":   gen_tok["output"],
        "tokens_judge_corr_input":  corr_tok["input"],
        "tokens_judge_corr_output": corr_tok["output"],
        "tokens_judge_faith_input": faith_tok["input"],
        "tokens_judge_faith_output":faith_tok["output"],
        "tokens_total_input":       total_in,
        "tokens_total_output":      total_out,
        "est_cost_usd":             est_cost,
    })
    time.sleep(60)

results_df = pd.DataFrame(results)
Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)
out_path = Path(CONFIG["results_dir"]) / "baseline_results.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved {len(results_df)} rows -> {out_path}")

Evaluating baseline:   0%|          | 0/20 [00:00<?, ?it/s]


Q1/20 [extractive]  AAPL  2023.0
  corr=0.00  faith=0.00  f1=0.00  tokens=3402in/135out  est=$0.00039


Evaluating baseline:   5%|▌         | 1/20 [01:03<19:57, 63.01s/it]


Q2/20 [extractive]  AAPL  2025.0
  corr=0.00  faith=1.00  f1=0.34  tokens=5003in/204out  est=$0.00058


Evaluating baseline:  10%|█         | 2/20 [02:06<18:55, 63.07s/it]

...


Q8/20 [paraphrased]  MSFT  2023.0
  corr=0.00  faith=0.00  f1=0.05  tokens=3274in/131out  est=$0.00038


Evaluating baseline:  40%|████      | 8/20 [08:23<12:34, 62.87s/it]


Q9/20 [paraphrased]  MSFT  2023.0
  corr=0.00  faith=0.00  f1=0.09  tokens=10685in/134out  est=$0.00112


Evaluating baseline:  40%|████      | 8/20 [09:26<14:09, 70.80s/it]


KeyboardInterrupt: 

## 9. Results

In [ ]:
print("=" * 60)
print(" BASELINE RAG — EVALUATION RESULTS")
print("=" * 60)

scored_c = results_df[results_df['llm_correctness_score'].notna()]
scored_f = results_df[results_df['llm_faithfulness_score'].notna()]
print(f'\nTotal questions    : {len(results_df)}')
print(f'Correctness judged : {len(scored_c)}')
print(f'Faithfulness judged: {len(scored_f)}')

print('\nOverall metrics:')
for col, label in [
    ('llm_correctness_score',  'LLM Correctness  '),
    ('llm_claims_covered',     'Claims Covered   '),
    ('llm_faithfulness_score', 'LLM Faithfulness '),
    ('token_f1',               'Token F1         '),
    ('numerical_accuracy',     'Numerical Accuracy'),
    ('decision_accuracy',      'Decision Accuracy'),
]:
    vals = results_df[col].dropna()
    if len(vals):
        print(f'  {label}: {vals.mean():.4f}  (n={len(vals)})')

print('\nBy question_type:')
type_cols = ['llm_correctness_score', 'llm_faithfulness_score', 'token_f1', 'decision_accuracy']
avail = [c for c in type_cols if c in results_df.columns]
if avail and 'question_type' in results_df.columns:
    print(results_df.groupby('question_type')[avail].mean().round(3).to_string())

if 'latency_sec' in results_df.columns:
    lat = results_df['latency_sec'].dropna()
    if len(lat):
        print(f'\nLatency: mean={lat.mean():.1f}s  median={lat.median():.1f}s  max={lat.max():.1f}s')

print('\nToken & cost summary:')
for col in ['tokens_generate_input', 'tokens_generate_output',
            'tokens_judge_corr_input', 'tokens_judge_corr_output',
            'tokens_judge_faith_input', 'tokens_judge_faith_output',
            'tokens_total_input', 'tokens_total_output']:
    if col in results_df.columns:
        print(f'  {col:35s}: {int(results_df[col].sum()):,}')
if 'est_cost_usd' in results_df.columns:
    print(f'  {"Total estimated cost (USD)":35s}: ${results_df["est_cost_usd"].sum():.4f}')

In [25]:
results_df

,question_id,question,question_type,expected_decision,expected_answer,predicted_answer,refused,correctness,faithfulness,claims_covered,n_chunks,corr_explanation
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,ANSWER,Apple's total net sales in fiscal year 2023 we...,"Unfortunately, the provided context does not c...",True,0.0,1.0,0.0,5,The candidate answer is incorrect because it d...
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",I cannot provide information on countries subj...,False,0.0,1.0,0.0,5,Candidate answer does not provide any informat...
2,3.0,What are the contingencies declared by Microso...,extractive,ANSWER,Microsoft declared a contingency related to IR...,I couldn't find any information about Microsof...,False,0.0,1.0,0.0,5,The candidate answer is incorrect because it d...
3,4.0,Which Microsoft business segment generated the...,extractive,ANSWER,Intelligent Cloud generated the highest revenu...,"Based on the provided context, I am unable to ...",True,0.0,1.0,0.0,5,The candidate answer is incorrect as it does n...
4,5.0,What risk does Nvidia mention that could affec...,extractive,ANSWER,Customers may delay adopting new architectures...,I couldn't find any information about Nvidia i...,False,0.0,1.0,0.0,5,The candidate answer is incorrect because it m...
5,26.0,What amount of revenue did Apple generate duri...,paraphrased,ANSWER,Apple's total net sales in fiscal year 2023 we...,I cannot provide information on Apple's revenu...,False,0.0,1.0,0.0,5,Candidate answer is completely missing the req...
6,27.0,Which regions were impacted by the newly annou...,paraphrased,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",I cannot provide information on the newly anno...,False,0.0,1.0,0.0,5,The candidate answer does not provide any info...
7,28.0,What potential financial obligations did Micro...,paraphrased,ANSWER,Microsoft declared a contingency related to IR...,"Based on the provided context, I couldn't find...",False,0.0,1.0,0.0,5,The candidate answer is incorrect because it d...
8,29.0,Which of Microsoft’s operating divisions contr...,paraphrased,ANSWER,Intelligent Cloud generated the highest revenu...,"Based on the provided context, I am unable to ...",True,0.0,1.0,0.0,5,The candidate answer is completely incorrect a...
9,30.0,What factor does Nvidia identify that could de...,paraphrased,ANSWER,Customers may delay adopting new architectures...,I couldn't find any information in the provide...,False,0.0,1.0,0.0,5,The candidate answer is completely off-topic a...
